### 今天要回答的問題

> **海表溫距平值越高，就代表海洋熱浪的分級越高嗎？**


本練習將使用 ODB Marine Heatwave API，取得單一地點的月尺度資料，並利用 Python：

1. 呼叫 API 並將 JSON 整理成 DataFrame  
2. API 輸出欄位 (SST、SST anomaly 與 MHW level)  
3. 畫出 距平值 與 海洋熱浪分級 時序圖  
4. 比較 距平值 與 海洋熱浪分級 的關係  

資料來源

- ODB MHW API 文件：  
  https://eco.odb.ntu.edu.tw/api/swagger/mhw
- ODB Marine Heatwaves 網頁：  
  https://eco.odb.ntu.edu.tw/pub/MHW/

熱浪等級對照表

| level | Marine Heatwave category |
|---:|---|
| -1 | 海冰（無距平值，不分級）|
| 0 | Non-MHW |
| 1 | Moderate |
| 2 | Strong |
| 3 | Severe |
| 4 | Extreme |

### 1. 載入套件

這次會使用：

- `requests`：向 API 發送請求
- `pandas`：整理與分析資料
- `matplotlib`：繪製時序圖

In [ ]:
import requests
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)

### 2. 設定查詢條件

先分析臺灣東北方、黑潮流經附近的一個格點：

- 經度：122.625°E
- 緯度：25.375°N
- 時間：2010-01 至 2025-12

ODB MHW API 的單點查詢沒有時間長度限制。

> 日期仍建議填寫每月第一天，例如 `2015-01-01`。

In [ ]:
# API endpoint
url = "https://eco.odb.ntu.edu.tw/api/mhw"

# 位置與時間
# 這裡預填臺灣東北方、黑潮流經附近的格點，請改成你想看的地點。
# 東經與北緯填正值，西經與南緯填負值；
# 例如南加州 La Jolla 外海（講義的範例地點）是 lon = -117.375, lat = 32.875
lon = 122.625   # 請自行修改
lat = 25.375

start_date = "2010-01-01"
end_date = "2025-12-01"

# append 希望 API 回傳的欄位
params = {
    "lon0": lon,
    "lat0": lat,
    "start": start_date,
    "end": end_date,
    "append": "sst,sst_anomaly,level",
}

params

### 3. 呼叫 API

`requests.get()` 會把查詢條件送到 ODB API。

`raise_for_status()` 可在 API 回傳錯誤狀態時立即顯示錯誤，而不是繼續處理不完整的資料。

In [ ]:
response = requests.get(url, params=params, timeout=60)

# API 出錯時會在回應內容裡說明原因，但 raise_for_status() 只看狀態碼，
# 不會把原因顯示出來。先印出來，才知道是座標超出範圍還是日期不對。
if not response.ok:
    try:
        print("API 回應內容：", response.json())
    except ValueError:
        print("API 回應內容：", response.text[:300])

response.raise_for_status()

print("Request URL:")
print(response.url)
print()
print("HTTP status:", response.status_code)

### 4. 將 JSON 整理成 DataFrame

API 通常會回傳由多筆紀錄組成的 JSON。

下方程式可以滿足：
- JSON 最外層直接是 list
- JSON 最外層是 dictionary，資料放在其中某個 list 欄位

In [ ]:
json_data = response.json()

if isinstance(json_data, list):
    records = json_data
elif isinstance(json_data, dict):
    # 嘗試尋找 dictionary 中第一個由多筆紀錄組成的 list
    list_values = [value for value in json_data.values() if isinstance(value, list)]
    if not list_values:
        raise ValueError(
            "API 回傳的是 dictionary，但沒有找到可轉成表格的 list。"
            f"\n回傳欄位：{list(json_data.keys())}"
        )
    records = list_values[0]
else:
    raise TypeError(f"無法辨識 API 回傳格式：{type(json_data)}")

df = pd.json_normalize(records)

# 空表的話，後面每一步都會丟出看不懂的錯誤，先在這裡擋下來
if df.empty:
    raise ValueError("API 回傳 0 筆資料。請檢查座標與日期範圍是否正確，"
                     "以及查詢期間是否落在資料涵蓋的年份內。")

print("資料筆數：", len(df))
print("欄位：", df.columns.tolist())
df.head()


### 5. 資料處理流程1｜認識資料

在畫圖以前，請先看看：

1. 這份資料共有幾筆？
2. 有哪些欄位？
3. 最早與最晚的日期分別是什麼時候？
4. 資料是否以一個月為間隔？

In [ ]:
# API 輸出欄位通常使用 date；此段 time 欄位也ok
date_candidates = ["date", "time"]
date_col = next((col for col in date_candidates if col in df.columns), None)

if date_col is None:
    raise KeyError(
        "找不到日期欄位。請檢查 df.columns，確認 API 實際使用的欄位名稱。"
    )

df[date_col] = pd.to_datetime(df[date_col])
df = df.sort_values(date_col).reset_index(drop=True)

print(f"資料筆數：{len(df)}")
print(f"時間範圍：{df[date_col].min().date()} 至 {df[date_col].max().date()}")

# 資料是否以一個月為間隔？
# 要用「我們要求的範圍」來比對。若改用「實際收到的最早與最晚日期」，
# 當 API 少給了頭尾的月份時，比對範圍會跟著縮短，結果永遠是「缺 0 個月」。
# 若 start 與 end 寫反了，API 會自行正規化，這裡也照做，否則會算出「要求 0 個月」
req_from, req_to = sorted([pd.Timestamp(start_date), pd.Timestamp(end_date)])
requested_months = pd.period_range(req_from.to_period("M"),
                                   req_to.to_period("M"), freq="M")
got_months = df[date_col].dt.to_period("M")
missing_months = requested_months.difference(got_months.unique())

print(f"要求 {len(requested_months)} 個月，實得 {len(df)} 筆；"
      f"缺 {len(missing_months)} 個月，重複 {int(got_months.duplicated().sum())} 筆")
if len(missing_months):
    print("缺少的月份：", [str(m) for m in missing_months[:12]])


def format_point(lon_value, lat_value):
    """寫成慣用的方位表示，例如 117.375°W, 32.875°N。

    講義的範例地點在南加州（La Jolla 約西經 117 度），
    若一律寫成 °E 就會把西經標成東經。
    """
    ew = "E" if lon_value >= 0 else "W"
    ns = "N" if lat_value >= 0 else "S"
    return f"{abs(lon_value)}°{ew}, {abs(lat_value)}°{ns}"


# API 會把座標貼到最近的 0.25 度格心，畫圖時要標實際拿到的格點，不是我們輸入的值
if {"lon", "lat"} <= set(df.columns):
    grid_lon, grid_lat = df["lon"].iloc[0], df["lat"].iloc[0]
    print(f"要求的座標 ({lon}, {lat}) -> 實際取得的格點 ({grid_lon}, {grid_lat})")
else:
    grid_lon, grid_lat = lon, lat

point_label = format_point(grid_lon, grid_lat)

print()
df.info()

### 5. 資料處理流程2｜檢查缺值與資料型態

分析前先確認重要欄位是否存在，並將數值欄位轉成 numeric。

(這裡只是，以防萬一，若你用別人家的 api)

In [ ]:
required_columns = ["sst", "sst_anomaly", "level"]
missing_columns = [col for col in required_columns if col not in df.columns]

if missing_columns:
    raise KeyError(
        f"缺少必要欄位：{missing_columns}\n"
        f"目前欄位：{df.columns.tolist()}"
    )

for col in required_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("各欄位缺值數：")
display(df[[date_col, *required_columns]].isna().sum().to_frame("missing_count"))

df[[date_col, *required_columns]].head()

### 5. 資料處理流程3｜這個地點能回答我們的問題嗎？

本課要比較「距平值」與「分級」，但有些地點根本沒有距平值可比，而且原因不只一種：

| 情況 | `sst` | `sst_anomaly` | `level` |
|---|---|---|---|
| 陸地 | 空 | 空 | 0 |
| 海冰 | 有值 | 空 | -1 |
| 有海溫但算不出距平值 | 有值 | 空 | 0 |
| 正常海域 | 有值 | 有值 | 0 ~ 4 |

要特別注意第一列和第三列：**`level = 0` 不一定代表「沒有海洋熱浪」**，
也可能代表「這裡根本判斷不出來」。距平值要拿當月海溫減去 1982–2011 年的
氣候平均，如果基準期在這個格點的資料不足，就算不出距平值，`level` 只好留 0。

畫出來的長條圖一樣是一整排 Non-MHW，但意義完全不同。所以在這裡判斷一次，
後面每一步就不必各自再猜。

In [ ]:
n_month = len(df)
n_sst = int(df["sst"].notna().sum())               # 有幾個月量得到海溫
n_anomaly = int(df["sst_anomaly"].notna().sum())   # 有幾個月算得出距平值
n_seaice = int((df["level"] == -1).sum())          # 有幾個月是海冰

print(f"共 {n_month} 個月：{n_sst} 個月有海溫，{n_anomaly} 個月有距平值，"
      f"{n_seaice} 個月是海冰")
print()

if n_sst == 0:
    print("這個格點沒有任何海溫資料，座標很可能落在陸地上。")
elif n_anomaly == 0 and n_seaice:
    print(f"這個格點有 {n_seaice} 個月是海冰，其餘 {n_month - n_seaice} 個月")
    print("雖然量得到海溫，也算不出距平值，整段期間都沒有可比的資料。")
elif n_anomaly == 0:
    print("這個格點有海溫，但整段期間都算不出距平值")
    print("（基準期 1982-2011 在這裡的資料可能不足）。")
    print("此時 level 一律是 0，那代表「無法判斷」，不是「沒有海洋熱浪」。")
elif n_seaice:
    print(f"海冰的 {n_seaice} 個月無法分級，其餘 {n_anomaly} 個月可以正常比較。")
else:
    print(f"{n_anomaly} 個月都有距平值，可以直接往下做。")

if n_anomaly == 0:
    print()
    print("下面的圖仍然會畫，但「比較距平值與分級」的練習要換一個座標才做得起來。")

### 6. 畫出 SST anomaly 時序

先觀察每個月相對於 1982–2011 氣候平均值

圖中的水平虛線表示 `anomaly = 0`。

In [ ]:
import matplotlib.dates as mdates

fig, ax = plt.subplots(figsize=(14, 4))

ax.plot(df[date_col], df["sst_anomaly"], linewidth=1.2)
ax.axhline(0, linestyle="--", linewidth=1)
# ===== x 軸設定 =====
# 每年一個主刻度
ax.xaxis.set_major_locator(mdates.YearLocator())

# 每六個月一個次刻度
ax.xaxis.set_minor_locator(mdates.MonthLocator(interval=6))

# 主刻度顯示年份
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

# 次刻度顯示月份
ax.xaxis.set_minor_formatter(mdates.DateFormatter('%b'))

# 旋轉月份避免重疊
ax.tick_params(axis='x', which='major', pad=22)
ax.tick_params(axis='x', which='minor', labelsize=8, rotation=90)

ax.set_title(f"Monthly SST Anomaly ({point_label})")
ax.set_xlabel("Time")
ax.set_ylabel("SST anomaly (°C)")
ax.grid(alpha=0.25)

plt.tight_layout()
plt.show()

### 思考

1. 有沒有哪一個年份或月份的 anomaly 特別高？
2. 是否有一段時間 anomaly 連續多月為正？

### 7. 畫出 Marine Heatwave level

`level` 是海洋熱浪分級，今天選擇用長條圖來做比較。

In [ ]:
import matplotlib.dates as mdates

# level = -1 代表該月為海冰。ODB 的資料處理有一道海冰遮罩，海洋熱浪的距平值
# 讀自已遮罩的資料，所以海冰月份有 sst 但沒有 sst_anomaly，也就無法分級。
level_colors = {
    -1: "#bcd6e8",  # 海冰
    0: "#f2f2f2",   # Non-MHW
    1: "#f5c268",   # Moderate
    2: "#ec6b1a",   # Strong
    3: "#cb3827",   # Severe
    4: "#7f1416",   # Extreme
}

level_names = {
    -1: "Sea ice",
    0: "Non-MHW",
    1: "Moderate",
    2: "Strong",
    3: "Severe",
    4: "Extreme",
}

fig, ax = plt.subplots(figsize=(14, 3.8))

# 沒對應到的 level（缺值、或日後新增的等級）給灰色。
# 這裡刻意不寫 int(v)：若寫了，1.5 會被當成 1 而上色成 Moderate，
# 但下一格的 .map() 找不到 1.5 會給灰色，同一筆資料在兩張圖就不一致。
colors = [level_colors.get(v, "#cccccc") for v in df["level"]]

ax.bar(
    df[date_col],
    df["level"],
    width=25,
    color=colors,
    edgecolor="black",
    linewidth=0.3,
)

# x 軸
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

ax.set_title(f"Monthly Marine Heatwave Level ({point_label})")
ax.set_xlabel("Time")
ax.set_ylabel("MHW level")

# y 軸只在資料裡真的有海冰時才往下延伸，否則一般海域會多出一條空白帶
shown_levels = [-1, 0, 1, 2, 3, 4] if n_seaice else [0, 1, 2, 3, 4]
ax.set_yticks(shown_levels, [level_names[v] for v in shown_levels])
ax.set_ylim(-1.4 if n_seaice else -0.1, 4.4)   # 沒有海冰時維持原本的下緣

ax.grid(axis="y", alpha=0.25)

# 沒有距平值時，這些長條不是海洋熱浪分級，要在圖上講明白
# （圖上的字用英文，matplotlib 的預設字型沒有中文字）
if n_anomaly == 0:
    ax.text(0.5, 0.55,
            "No usable SST anomaly at this grid point.\n"
            "level = 0 here means 'cannot be determined', not 'no marine heatwave'.",
            ha="center", va="center", transform=ax.transAxes, fontsize=12,
            bbox=dict(facecolor="white", alpha=0.92, edgecolor="#cc0000"))

plt.tight_layout()
plt.show()

### 8. 將 SST anomaly 與 MHW level 放在同一張圖比較

灰色折線與圓點的位置表示每個月的 SST anomaly。

圓點顏色表示該月份的 Marine Heatwave level：

- 白色：Non-MHW
- 黃色：Moderate
- 橘色：Strong
- 紅色：Severe
- 深紅色：Extreme

請觀察：

1. anomaly 為正的月份，是否一定被判定為 MHW？
2. anomaly 最大的月份，是否一定具有最高的 category？
3. 相近的 anomaly，是否可能對應不同的 category？


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.lines import Line2D

# 這一格重新寫一次顏色與名稱，這樣單獨重跑這一格也不會出錯
level_colors = {
    -1: "#bcd6e8",  # 海冰
    0: "#ffffff",   # Non-MHW
    1: "#f5c268",   # Moderate
    2: "#ec6b1a",   # Strong
    3: "#cb3827",   # Severe
    4: "#7f1416",   # Extreme
}

level_names = {
    -1: "Sea ice",
    0: "Non-MHW",
    1: "Moderate",
    2: "Strong",
    3: "Severe",
    4: "Extreme",
}

# 根據每筆資料的 level 指定圓點顏色。沒對應到的給灰色，
# 否則 map() 會產生 NaN，scatter 會直接丟出 ValueError。
point_colors = df["level"].map(level_colors).fillna("#cccccc")

fig, ax = plt.subplots(figsize=(14, 5))

# SST anomaly 折線
ax.plot(
    df[date_col],
    df["sst_anomaly"],
    linewidth=1,
    color="gray",
    zorder=1,
)

# 每個月份加上圓點，顏色代表 MHW level
ax.scatter(
    df[date_col],
    df["sst_anomaly"],
    c=point_colors,
    s=45,
    edgecolors="black",
    linewidths=0.4,
    zorder=2,
)

# anomaly = 0 參考線
ax.axhline(
    0,
    linestyle="--",
    linewidth=1,
    color="gray",
)

# x 軸每 6 個月顯示一次
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
plt.setp(ax.get_xticklabels(), rotation=45, ha="right")

ax.set_title(
    f"Monthly SST Anomaly and MHW Level ({point_label})"
)
ax.set_xlabel("Time")
ax.set_ylabel("SST anomaly (°C)")
ax.grid(alpha=0.25)

# 圖例只列出「資料裡有、而且畫得出來」的等級。海冰月份沒有距平值，
# 散點沒有 y 座標，列進圖例反而讓人以為圖上找得到。
drawable = sorted(set(level_names)
                  & set(df.loc[df["sst_anomaly"].notna(), "level"].dropna().astype(int)))

legend_handles = [
    Line2D(
        [0],
        [0],
        marker="o",
        linestyle="none",
        markerfacecolor=level_colors[level],
        markeredgecolor="black",
        markeredgewidth=0.4,
        markersize=8,
        label=level_names[level],
    )
    for level in drawable
]

if legend_handles:
    ax.legend(
        handles=legend_handles,
        title="MHW category",
        loc="upper left",
        ncol=len(legend_handles),
    )

plt.tight_layout()
plt.show()

### 為什麼距平值高，分級不一定高？

關鍵在於**門檻不是固定的數字**。依 Hobday et al. (2018)，分級看的是
「距平值是門檻的幾倍」：1–2 倍為 Moderate、2–3 倍 Strong、3–4 倍 Severe、
4 倍以上 Extreme。

而門檻本身會隨**地點**和**月份**改變。ODB 採用 Jacox et al. (2020) 的月尺度定義，
某個月的門檻是由 1982–2011 年間**該月與前後各一個月**的距平值取第 90 百分位數而來
（判斷 2 月就綜合 1、2、3 月）。所以同一個地點，1 月和 10 月的門檻可以差很多。

這帶來兩個可以自己驗證的現象：

- **同一個曆月**之內，距平值越高，分級只會相同或更高，不會反過來。
- **跨曆月**比較時，距平值較低的月份反而可能分級較高，因為它的門檻比較低。

所以「這個月比較熱」和「這個月是比較嚴重的海洋熱浪」是兩件不同的事。

### 挑戰｜用程式回答問題

請不要只靠目測，利用 DataFrame 找出：

1. anomaly 最大的月份  
2. level 最高的月份  
3. 兩者是否為同一個月份  
4. 是否存在 anomaly > 0 但 level = 0 的月份

In [ ]:
# 沒有距平值的地點做不了這一題，前面已經判斷過了，這裡直接沿用結論
if n_anomaly == 0:
    print("這個格點沒有 SST 距平值，這一題要換一個座標才做得起來。")
else:
    # anomaly 最大的月份
    max_anomaly_row = df.loc[df["sst_anomaly"].idxmax()]

    print("Anomaly 最大的月份")
    display(
        max_anomaly_row[
            [date_col, "sst", "sst_anomaly", "level"]
        ].to_frame("value")
    )

In [ ]:
# level 最高的所有月份
max_level = df["level"].max()

highest_level_months = df.loc[
    df["level"] == max_level,
    [date_col, "sst", "sst_anomaly", "level"],
]

print("資料中的最高 level：", int(max_level))
print("達到最高 level 的月份數：", len(highest_level_months))
display(highest_level_months)


In [ ]:
# 比較 anomaly 最大月份與最高 level
if n_anomaly == 0:
    print("沒有距平值，這一題略過。")
else:
    max_anomaly_date = max_anomaly_row[date_col]
    max_anomaly_level = max_anomaly_row["level"]

    same_as_highest_level = max_anomaly_level == max_level

    print("Anomaly 最大月份：", max_anomaly_date.date())
    print("該月份的 level：", int(max_anomaly_level))
    print("整段資料的最高 level：", int(max_level))
    print()
    print("Anomaly 最大的月份是否達到最高 level？", same_as_highest_level)

In [ ]:
# 找出 anomaly 為正，但沒有被判定為 MHW 的月份
positive_but_not_mhw = df.loc[
    (df["sst_anomaly"] > 0) & (df["level"] == 0),
    [date_col, "sst", "sst_anomaly", "level"],
].copy()

print("Anomaly > 0，但 level = 0 的月份數：", len(positive_but_not_mhw))
display(positive_but_not_mhw.head(20))
